## Rotina para baixas dados mais atuais da estação Meteorológica e previsão numérica

**Objetivo:** Baixar os dados mais recentes da estação Meteorológica e previsão atmosférica

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

import openmeteo_requests

import requests_cache
from retry_requests import retry

from dotenv import load_dotenv
import os

### 1. OPENDAQ

In [2]:
load_dotenv()

API_KEY = os.getenv("OPENAQ_API_KEY")

headers = {
    "X-API-Key": API_KEY
}

BASE_URL = "https://api.openaq.org/v3"

In [3]:
# informações do ponto
lat = -23.6473149
lon = -46.6643635

In [4]:
# Data de hoje (UTC)
today = datetime.now(timezone.utc)

# Ontem
date_to = today - timedelta(days=1)

# 30 dias antes de ontem
date_from = date_to - timedelta(days=29)

date_to_str = date_to.strftime("%Y-%m-%dT23:59:59Z")
date_from_str = date_from.strftime("%Y-%m-%dT00:00:00Z")

print(date_from_str)
print(date_to_str)

2026-06-02T00:00:00Z
2026-07-01T23:59:59Z


In [5]:
location_id = 6139516

url = f"{BASE_URL}/locations/{location_id}/sensors"

r = requests.get(url, headers=headers)
r.raise_for_status()

sensors = pd.json_normalize(r.json()["results"])

In [6]:
def baixa_sensor(sensor_id, date_from, date_to, api_key):

    headers = {
        "X-API-Key": api_key
    }

    base_url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"

    page = 1
    limit = 1000

    dados = []

    while True:

        params = {
            "datetime_from": date_from_str,
            "datetime_to": date_to_str,
            "limit": limit,
            "page": page
        }

        r = requests.get(base_url, headers=headers, params=params)
        r.raise_for_status()

        js = r.json()

        results = js.get("results", [])

        if len(results) == 0:
            break

        dados.extend(results)

        found = js.get("meta", {}).get("found", 0)

        if page * limit >= found:
            break

        page += 1

    df = pd.json_normalize(dados)

    return df

In [7]:
dfs = {}

for _, row in sensors.iterrows():

    sensor_id = row["id"]
    nome = row["parameter.name"]

    print(f"Baixando: {nome}")

    try:

        df_tmp = baixa_sensor(
            sensor_id=sensor_id,
            date_from=date_from_str,
            date_to=date_to_str,
            api_key=API_KEY
        )

        dfs[nome] = df_tmp

    except Exception as e:

        print(f"Erro em {nome}: {e}")

Baixando: pm1


Baixando: pm25
Baixando: relativehumidity
Baixando: temperature
Baixando: um003


In [8]:
# ORGANIZA TODAS AS SÉRIES

serie_final = pd.DataFrame()

for nome, df_tmp in dfs.items():

    if len(df_tmp) == 0:
        continue

    df_aux = pd.DataFrame()


    df_aux["time"] = pd.to_datetime(
        df_tmp["period.datetimeFrom.local"]
    )


    df_aux[nome] = df_tmp["value"]


    df_aux = df_aux.set_index("time")


    df_aux = df_aux[~df_aux.index.duplicated(keep="first")]


    if serie_final.empty:

        serie_final = df_aux

    else:

        serie_final = serie_final.join(
            df_aux,
            how="outer"
        )

In [9]:
# ORDENA PELO TEMPO
serie_final = serie_final.sort_index()

date_str = pd.to_datetime(date_to).strftime("%Y%m%d")


arquivo_saida = f"../data/operational/openaq_location_6139516_{date_str}.csv"

serie_final.to_csv(
   arquivo_saida,
   sep=",",
   decimal=".",
   encoding="utf-8"
)

print(f"Arquivo salvo: {arquivo_saida}")

Arquivo salvo: ../data/operational/openaq_location_6139516_20260701.csv


### 2. Meteo

In [10]:
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


url = "https://archive-api.open-meteo.com/v1/archive"

In [11]:
date_to_str = date_to.strftime("%Y-%m-%d")
date_from_str = date_from.strftime("%Y-%m-%d")

print(date_from_str)
print(date_to_str)

2026-06-02
2026-07-01


In [12]:
params = {
    "latitude": -23.6473149,
    "longitude": -46.6643635,
    "start_date": date_from_str,
    "end_date": date_to_str,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
    ],
}

In [13]:
responses = openmeteo.weather_api(url, params = params)


response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)

Coordinates: -23.655536651611328°N -46.7088623046875°E
Elevation: 793.0 m asl
Timezone difference to GMT+0: 0s


In [14]:
# ORGANIZA DATAFRAME
hourly_dataframe = hourly_dataframe.rename(columns={"date": "time"})

# converter UTC para horário local de São Paulo
hourly_dataframe["time"] = (
    pd.to_datetime(hourly_dataframe["time"], utc=True)
    .dt.tz_convert("America/Sao_Paulo")
)

hourly_dataframe = hourly_dataframe.set_index("time")
hourly_dataframe = hourly_dataframe.sort_index()

In [15]:
# SALVAR CSV

date_str = pd.to_datetime(date_to).strftime("%Y%m%d")

arquivo_saida = f"../data/operational/openmeteo_operacional_{date_str}.csv"

hourly_dataframe.to_csv(
   arquivo_saida,
   encoding="utf-8"
)

print(f"Arquivo salvo: {arquivo_saida}")

Arquivo salvo: ../data/operational/openmeteo_operacional_20260701.csv
